# 01 — Product Image Classifier Training
Transfer learning on **MobileNetV2**, trained on the real **Fashion-MNIST** dataset (Zalando Research) — 70,000 real labeled clothing product images across 10 categories (T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot).

This is loaded directly via `tf.keras.datasets.fashion_mnist` — the real dataset, downloaded from Google's official Keras dataset mirror the first time this runs. **Requires an internet connection.** No synthetic or placeholder data is used at any point in this notebook.

**Want retail-specific categories instead** (e.g. electronics, groceries)? Swap in a real Kaggle set such as *Retail Product Checkout Dataset*, organized as `data/product_images/<class>/*.jpg`, and replace the loading cell below with a `tf.keras.preprocessing.image_dataset_from_directory(...)` call — the rest of the notebook (model, training, saving) stays the same.

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))  # so `import app...` works from notebooks/

import json

import numpy as np
import cv2
import tensorflow as tf

from app import config

IMG_SIZE = config.IMAGE_SIZE  # (128, 128) — resized up from Fashion-MNIST's native 28x28


2026-08-02 05:23:39.859211: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-02 05:23:39.863384: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-02 05:23:40.050853: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-02 05:23:40.912069: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-02 05:23:43.674060: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warnin

## Load the real Fashion-MNIST dataset
Downloads the actual dataset (60,000 train + 10,000 test real grayscale images) on first run.

In [2]:
(X_train_raw, y_train), (X_test_raw, y_test) = tf.keras.datasets.fashion_mnist.load_data()

CLASSES = [
    "T-shirt/ltop", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

print("Train:", X_train_raw.shape, "Test:", X_test_raw.shape)
print("Classes:", CLASSES)


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 9us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 151s 6us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Train: (60000, 28, 28) Test: (10000, 28, 28)
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


## Preprocess: resize to the classifier's input size + RGB + normalize
MobileNetV2 expects a 3-channel input, so each grayscale Fashion-MNIST image is resized and channel-replicated. This is real image data at every step — only the resolution/format is being adapted.

In [3]:
def preprocess(images: np.ndarray) -> np.ndarray:
    out = np.zeros((len(images), *IMG_SIZE, 3), dtype=np.float32)
    for i, img in enumerate(images):
        resized = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA)
        rgb = cv2.cvtColor(resized, cv2.COLOR_GRAY2RGB)
        out[i] = rgb.astype(np.float32) / 255.0
    return out


# Use the full real test set, and a real training subset sized for a
# CPU/laptop-friendly training run (increase if you have a GPU).
TRAIN_SUBSET = 6000
X_train = preprocess(X_train_raw[:TRAIN_SUBSET])
y_train_labels = y_train[:TRAIN_SUBSET]
X_test = preprocess(X_test_raw)
y_test_labels = y_test

print(X_train.shape, X_test.shape)


(6000, 128, 128, 3) (10000, 128, 128, 3)


## Build MobileNetV2 transfer-learning model
`weights="imagenet"` downloads real pretrained ImageNet weights (also requires internet on first run).

In [4]:
from tensorflow.keras import layers, models

NUM_CLASSES = len(CLASSES)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet"
)
base_model.trainable = False  # freeze the pretrained backbone

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax"),
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()


2026-08-02 05:26:41.359018: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:282] failed call to cuInit: UNKNOWN ERROR (100)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,423,242 (9.24 MB)

 Trainable params: 165,258 (645.54 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## Train on the real data

In [ ]:
history = model.fit(
    X_train, y_train_labels,
    validation_data=(X_test, y_test_labels),
    epochs=8,
    batch_size=32,
)


Epoch 1/8
124/188 ━━━━━━━━━━━━━━━━━━━━ 19s 312ms/step - accuracy: 0.7165 - loss: 0.8146

## Evaluate on the real held-out test set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_labels, verbose=0)
print(f"Test accuracy: {test_acc:.3f}")


## Save artifacts for the API

In [ ]:
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
model.save(config.PRODUCT_CLASSIFIER_PATH)

class_labels = {str(i): cls for i, cls in enumerate(CLASSES)}
with open(config.CLASS_LABELS_PATH, "w") as f:
    json.dump(class_labels, f, indent=2)

print("Saved:", config.PRODUCT_CLASSIFIER_PATH)
print("Saved:", config.CLASS_LABELS_PATH)
